# Analysis of df_wide: LLM vs Expert Agreement Patterns

**Assumes:** `df_wide` already loaded in global environment

Focus: **Batch_2** vs Expert and **Domain_Shot** vs Expert

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Overall agreement: % matches (including both NaN = match)
def agree_pct(col1, col2):
    matches = ((col1 == col2) | (col1.isna() & col2.isna())).sum()
    return (matches / len(col1)) * 100

b2_agree = agree_pct(df_wide['Result_Expert'], df_wide['Result_Batch_2'])
ds_agree = agree_pct(df_wide['Result_Expert'], df_wide['Result_Domain_Shot'])

print("OVERALL AGREEMENT")
print(f"Batch_2:      {b2_agree:.1f}%")
print(f"Domain_Shot:  {ds_agree:.1f}%")
print(f"Difference:   {b2_agree - ds_agree:+.1f}%")
print(f"\nMissing data: Expert={df_wide['Result_Expert'].isna().sum()}, "
      f"B2={df_wide['Result_Batch_2'].isna().sum()}, "
      f"DS={df_wide['Result_Domain_Shot'].isna().sum()}")

In [ ]:
# Agreement by domain (bar chart)
domains = sorted([int(d) for d in df_wide['Bias domain'].unique()])
b2_by_domain = []
ds_by_domain = []

for domain in domains:
    mask = df_wide['Bias domain'] == str(domain)
    b2_by_domain.append(agree_pct(df_wide.loc[mask, 'Result_Expert'], 
                                   df_wide.loc[mask, 'Result_Batch_2']))
    ds_by_domain.append(agree_pct(df_wide.loc[mask, 'Result_Expert'], 
                                   df_wide.loc[mask, 'Result_Domain_Shot']))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(domains))
width = 0.35
ax.bar(x - width/2, b2_by_domain, width, label='Batch_2', alpha=0.8)
ax.bar(x + width/2, ds_by_domain, width, label='Domain_Shot', alpha=0.8)
ax.set_xlabel('Bias Domain')
ax.set_ylabel('Agreement (%)')
ax.set_title('Agreement by Domain')
ax.set_xticks(x)
ax.set_xticklabels(domains)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("Agreement % by Domain:")
for i, d in enumerate(domains):
    print(f"  Domain {d}: B2={b2_by_domain[i]:.0f}% | DS={ds_by_domain[i]:.0f}%")

In [ ]:
# Overestimate vs Underestimate by domain
def count_bias(col1, col2):
    mask = col1.notna() & col2.notna()
    over = ((col1[mask] == 'High') & (col2[mask].isin(['Low', 'Medium']))).sum()
    under = ((col1[mask].isin(['Low', 'Medium'])) & (col2[mask] == 'High')).sum()
    return over, under

summary = []
for domain in domains:
    mask = df_wide['Bias domain'] == str(domain)
    exp = df_wide.loc[mask, 'Result_Expert']
    b2 = df_wide.loc[mask, 'Result_Batch_2']
    ds = df_wide.loc[mask, 'Result_Domain_Shot']
    
    b2_over, b2_under = count_bias(b2, exp)
    ds_over, ds_under = count_bias(ds, exp)
    
    summary.append({
        'Domain': domain,
        'B2_Over': b2_over, 'B2_Under': b2_under,
        'DS_Over': ds_over, 'DS_Under': ds_under
    })

df_summary = pd.DataFrame(summary)
print("\nOverestimate/Underestimate Counts:")
print(df_summary.to_string(index=False))
print("\n(Over = LLM says High when Expert says Low/Medium)")
print("(Under = LLM says Low/Medium when Expert says High)")

In [ ]:
# How well do LLMs classify each risk level?
risk_levels = sorted([x for x in df_wide['Result_Expert'].unique() if pd.notna(x)])

print("\nAccuracy by Risk Level (Expert says):")
for risk in risk_levels:
    mask = df_wide['Result_Expert'] == risk
    b2_match = ((df_wide.loc[mask, 'Result_Expert'] == df_wide.loc[mask, 'Result_Batch_2']) | 
                (df_wide.loc[mask, 'Result_Expert'].isna() & df_wide.loc[mask, 'Result_Batch_2'].isna())).sum()
    ds_match = ((df_wide.loc[mask, 'Result_Expert'] == df_wide.loc[mask, 'Result_Domain_Shot']) | 
                (df_wide.loc[mask, 'Result_Expert'].isna() & df_wide.loc[mask, 'Result_Domain_Shot'].isna())).sum()
    total = mask.sum()
    print(f"  {risk:15} n={total:2d}  B2: {b2_match}/{total} ({100*b2_match/total:.0f}%)  "
          f"DS: {ds_match}/{total} ({100*ds_match/total:.0f}%)")

In [ ]:
# Which papers are hardest to assess?
papers = sorted([int(p) for p in df_wide['Paper no.'].unique()])

paper_agree = []
for paper in papers:
    mask = df_wide['Paper no.'] == str(paper)
    b2 = agree_pct(df_wide.loc[mask, 'Result_Expert'], df_wide.loc[mask, 'Result_Batch_2'])
    ds = agree_pct(df_wide.loc[mask, 'Result_Expert'], df_wide.loc[mask, 'Result_Domain_Shot'])
    avg = (b2 + ds) / 2
    paper_agree.append({'Paper': paper, 'B2%': b2, 'DS%': ds, 'Avg%': avg})

df_paper = pd.DataFrame(paper_agree).sort_values('Avg%', ascending=False).reset_index(drop=True)

print("\nPaper Difficulty Ranking (by avg agreement):")
print("Rank 1 = Easiest, Rank 10 = Hardest")
for idx, row in df_paper.iterrows():
    print(f"  Rank {idx+1:2d}: Paper {int(row['Paper']):2d}  B2={row['B2%']:5.0f}%  "
          f"DS={row['DS%']:5.0f}%  Avg={row['Avg%']:5.0f}%")

In [ ]:
# Confusion matrices: what misclassifications happen?
categories = ['High', 'Medium', 'Low', 'Not Applicable']

def confusion_matrix(expert, llm):
    mask = expert.notna() & llm.notna()
    matrix = pd.DataFrame(0, index=categories, columns=categories)
    for e, l in zip(expert[mask], llm[mask]):
        if e in categories and l in categories:
            matrix.loc[l, e] += 1
    return matrix

cm_b2 = confusion_matrix(df_wide['Result_Expert'], df_wide['Result_Batch_2'])
cm_ds = confusion_matrix(df_wide['Result_Expert'], df_wide['Result_Domain_Shot'])

print("\nConfusion Matrix: Batch_2 vs Expert")
print("(Rows=Batch_2 predicts, Cols=Expert says)")
print(cm_b2)
print(f"\nConfusion Matrix: Domain_Shot vs Expert")
print("(Rows=Domain_Shot predicts, Cols=Expert says)")
print(cm_ds)
print("\n(Diagonal = correct | Off-diagonal = errors)")